# CivBot — Colab fine-tuning notebook

Trains a LoRA adapter on `mistralai/Mistral-7B-Instruct-v0.3` using the Civ VI leader-dialogue dataset, merges it, and exports a **4-bit GGUF** (~4 GB) you can run locally with Ollama or llama.cpp on CPU.

**Before Run All:**
1. Runtime → Change runtime type → **T4 GPU** (free tier is fine).
2. Have your Hugging Face token ready (Mistral is gated — accept the model card on HF first).
3. Be ready to upload `civ6_finetune_dataset.jsonl` when prompted.

Total runtime on a T4: ~30–45 min training + ~10 min export.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q -U \
  "transformers>=4.44.0" \
  "peft>=0.11.0" \
  "accelerate>=0.33.0" \
  "bitsandbytes>=0.43.0" \
  "datasets>=2.20.0" \
  "sentencepiece>=0.2.0" \
  "safetensors>=0.4.3"

## 3. Hugging Face login

Mistral-7B-Instruct is a gated model. Visit https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3 once and click **Agree and access**, then paste a token from https://huggingface.co/settings/tokens (a read token is enough).

In [ ]:
from huggingface_hub import login
login()  # paste token in the prompt

## 4. Upload the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick civ6_finetune_dataset.jsonl
DATA_PATH = next(iter(uploaded.keys()))
print('Uploaded:', DATA_PATH)

## 5. Train (QLoRA)

In [ ]:
import json, torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          DataCollatorForLanguageModeling, Trainer, TrainingArguments)

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'
OUTPUT_DIR = 'civbot-lora'
MAX_LEN = 512
EPOCHS = 3

torch.manual_seed(42)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

rows = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
raw = Dataset.from_list(rows).shuffle(seed=42)

def format_example(ex):
    user = f"{ex['instruction'].strip()}\n\n{ex.get('input','').strip()}".strip()
    return f"<s>[INST] {user} [/INST] {ex['output'].strip()}</s>"

def tokenize(batch):
    texts = [format_example({'instruction': i, 'input': inp, 'output': o})
             for i, inp, o in zip(batch['instruction'], batch['input'], batch['output'])]
    return tokenizer(texts, truncation=True, max_length=MAX_LEN, padding=False)

tokenized = raw.map(tokenize, batched=True, remove_columns=raw.column_names)
split = tokenized.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'train={len(train_ds)} eval={len(eval_ds)}')

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb,
                                             device_map='auto', torch_dtype=torch.bfloat16)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                  task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj',
                                  'gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=2, per_device_eval_batch_size=2,
    gradient_accumulation_steps=8, gradient_checkpointing=True,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03,
    optim='paged_adamw_8bit', bf16=True,
    logging_steps=10, save_strategy='epoch', eval_strategy='epoch',
    save_total_limit=1, report_to='none', seed=42)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                  data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False))
trainer.train()
trainer.model.save_pretrained(f'{OUTPUT_DIR}/final')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/final')
print('Adapter saved.')

## 6. Quick sanity check

In [ ]:
from peft import PeftModel
test_model = PeftModel.from_pretrained(model, f'{OUTPUT_DIR}/final').eval()
prompt = '<s>[INST] Generate dynamic dialogue for the following Civilization VI leader based on the game state.\n\nLeader: Cleopatra\nState: Greeting, Hostile [/INST]'
inputs = tokenizer(prompt, return_tensors='pt').to(test_model.device)
out = test_model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.8, top_p=0.9)
print(tokenizer.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True))

## 7. Merge adapter into base (fp16)

Reload base in fp16 (not 4-bit) so merging is mathematically correct.

In [ ]:
import gc
del model, trainer, test_model
gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map='cpu')
merged = PeftModel.from_pretrained(base, f'{OUTPUT_DIR}/final')
merged = merged.merge_and_unload()
merged.save_pretrained('civbot-merged', safe_serialization=True)
tokenizer.save_pretrained('civbot-merged')
print('Merged model saved to ./civbot-merged')

## 8. Convert to GGUF and quantize to Q4_K_M

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF > /dev/null
!cmake --build llama.cpp/build --config Release -j --target llama-quantize > /dev/null

In [ ]:
!python llama.cpp/convert_hf_to_gguf.py civbot-merged --outfile civbot-f16.gguf --outtype f16
!./llama.cpp/build/bin/llama-quantize civbot-f16.gguf civbot-q4_k_m.gguf Q4_K_M
!ls -lh civbot-*.gguf

## 9. Download the artifact

`civbot-q4_k_m.gguf` is the only file you need locally (~4 GB). Right-click it in the file pane → Download, **or** run the cell below.

If Colab's browser download chokes on 4 GB, use the Drive cell that follows.

In [ ]:
from google.colab import files
files.download('civbot-q4_k_m.gguf')

In [ ]:
# Alternative: copy to Google Drive, download from there.
from google.colab import drive
drive.mount('/content/drive')
!cp civbot-q4_k_m.gguf /content/drive/MyDrive/
print('Copied to Drive root.')

## 10. (Optional) Also save the LoRA adapter

Tiny (~40 MB) — useful as a portable artifact for re-merging onto a different base or for the project report.

In [ ]:
!zip -qr civbot-lora-final.zip civbot-lora/final
from google.colab import files
files.download('civbot-lora-final.zip')